# Part 2 — Job Postings Analysis: Role Categorization & Requirements Extraction

**Pipeline:** load 25 job postings → classify each into a broad domain (Step 2) →
extract skills / education / experience as structured JSON (Step 3) → run over the
whole set (Step 4) → merge back into the DataFrame (Step 5).

**Design choices worth noting in the write-up:**
- Two separate chains (classification and extraction) rather than one mega-prompt, so each
  sub-task can be prompted, tested and debugged independently.
- Classification uses a few-shot prompt + a constrained label set + fuzzy post-normalisation,
  so the model can never invent a category outside the taxonomy.
- Extraction uses a Pydantic schema with `with_structured_output`, so the result is a typed
  object instead of free text that needs regex parsing. A parser-based fallback is included
  for providers that don't support tool calling.
- Every call is wrapped in retry + backoff, and missing fields default to `"Not specified"`.

In [ ]:
# Install dependencies (uncomment on first run)
# !pip install -q langchain langchain-core langchain-ollama pandas pydantic requests
#
# And on the Ollama side, in a terminal:
#   ollama serve
#   ollama pull qwen2.5:7b

In [1]:
import os
import re
import json
import time
import difflib
from typing import List

import pandas as pd
from pydantic import BaseModel, Field

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

## Preflight — is Ollama up, and which models are pulled?

Run `ollama serve` in a terminal first (the desktop app does this for you). This cell fails
loudly now rather than 20 postings into Step 4.

In [2]:
import requests

OLLAMA_HOST = "http://localhost:11434"

try:
    tags = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=5).json()
    available = [m["name"] for m in tags.get("models", [])]
    print("Ollama is running. Models available locally:")
    for m in available:
        print("  -", m)
except Exception as e:
    available = []
    print(f"Could not reach Ollama at {OLLAMA_HOST} ({type(e).__name__}: {e})")
    print("Start it with `ollama serve`, then pull a model, e.g. `ollama pull qwen2.5:7b`")

Ollama is running. Models available locally:
  - llama3.2:latest


## LLM setup (local Ollama)

Three settings here do real work and are worth a line each in the write-up:

- **`num_ctx=8192`** — this is the one that silently ruins runs. Ollama's default context is
  2048 tokens; a full job description plus the few-shot prompt overflows that, and the
  *beginning* of the prompt gets dropped — which is where the instructions live. The model then
  returns plausible-looking garbage with no error raised anywhere. Set it explicitly.
- **`temperature=0`** — extraction and classification are deterministic tasks. Sampling would
  make the same posting produce different labels across runs and make results unreproducible.
- **`keep_alive="30m"`** — keeps the weights resident in RAM/VRAM between the 50 calls instead
  of reloading the model on each one.

**Model choice.** Step 3 needs reliable JSON, which on small local models is the limiting factor.
`qwen2.5:7b` is the safest pick for structured extraction; `llama3.1:8b` also works. Anything
below ~7B tends to produce malformed JSON and will fall back to the parser path. Use whatever
`ollama list` showed above.

In [3]:
from langchain_ollama import ChatOllama

OLLAMA_MODEL = "llama3.2:latest"     # <- set to a model from the list above

llm = ChatOllama(
    model=OLLAMA_MODEL,
    temperature=0,
    num_ctx=8192,        # MUST be set — default 2048 silently truncates the prompt
    keep_alive="30m",    # don't reload weights between calls
    base_url=OLLAMA_HOST,
)

# Warm-up call: loads the model into memory so the first real call isn't misleadingly slow,
# and confirms the model name is valid.
_t = time.time()
print(llm.invoke("Reply with the single word: ready").content.strip())
print(f"warm-up took {time.time() - _t:.1f}s — expect roughly this per call in Step 4")

ready
warm-up took 3.5s — expect roughly this per call in Step 4


In [4]:
# Classification only ever needs a handful of tokens back, so cap num_predict to stop a chatty
# local model writing a paragraph of justification we'd only throw away.
#
# NOTE: this must be a separate ChatOllama instance, NOT llm.bind(num_predict=16).
# `.bind()` forwards kwargs straight into the ollama client's Client.chat() call, and chat()
# has no num_predict parameter — it belongs inside the `options` dict that ChatOllama builds
# from its own constructor fields. Binding it raises:
#   TypeError: Client.chat() got an unexpected keyword argument 'num_predict'
llm_classify = ChatOllama(
    model=OLLAMA_MODEL,
    temperature=0,
    num_ctx=8192,
    num_predict=16,      # sampling option — set on the constructor, never via bind()
    keep_alive="30m",
    base_url=OLLAMA_HOST,
)

In [5]:
# Small helper: retry with backoff. Local models fail differently from APIs — no rate limits,
# but malformed JSON and occasional timeouts on long descriptions. base_delay is short since
# there's no quota to back off from.

CALL_FAILURES = []   # records any call that exhausted its retries, so silent degradation is visible


def run_with_retry(fn, *args, retries=3, base_delay=1, label="", **kwargs):
    last_err = None
    for attempt in range(retries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last_err = e
            wait = base_delay * (2 ** attempt)
            print(f"    [{label}] attempt {attempt + 1} failed ({type(e).__name__}: {e}) — retrying in {wait}s")
            time.sleep(wait)
    print(f"    [{label}] giving up after {retries} attempts: {last_err}")
    CALL_FAILURES.append({"label": label, "error": f"{type(last_err).__name__}: {last_err}"})
    return None

## Step 1 — Load the dataset

Kaggle: *Job Title and Job Description Dataset* (`job_title_des.csv`). Column names differ a
little between mirrors of this dataset, so the loader auto-detects the title/description
columns instead of hard-coding them.

In [6]:
DATA_PATH = "job_title_des.csv"   # <- change to your local path
N_POSTINGS = 25

df_raw = pd.read_csv(DATA_PATH)
print("Raw shape:", df_raw.shape)
print("Columns:", list(df_raw.columns))


def detect_columns(frame: pd.DataFrame):
    """Find the title and description columns regardless of naming convention."""
    title_col = desc_col = None
    for col in frame.columns:
        key = col.strip().lower().replace(" ", "_")
        if title_col is None and "title" in key:
            title_col = col
        if desc_col is None and ("description" in key or key in {"job_des", "des", "jd"}):
            desc_col = col
    if title_col is None or desc_col is None:
        raise ValueError(f"Could not detect title/description columns in {list(frame.columns)}")
    return title_col, desc_col


TITLE_COL, DESC_COL = detect_columns(df_raw)
print(f"Using title column = '{TITLE_COL}', description column = '{DESC_COL}'")

# Keep the first 25 postings, drop empties, standardise names
df = (
    df_raw[[TITLE_COL, DESC_COL]]
    .dropna()
    .rename(columns={TITLE_COL: "Job_Title", DESC_COL: "Job_Description"})
    .head(N_POSTINGS)
    .reset_index(drop=True)
)

print("\nWorking shape:", df.shape)
df.head()

Raw shape: (2277, 3)
Columns: ['Unnamed: 0', 'Job Title', 'Job Description']
Using title column = 'Job Title', description column = 'Job Description'

Working shape: (25, 2)


,Job_Title,Job_Description
0,Flutter Developer,We are looking for hire experts flutter developer. So you are eligible this post then apply your resume.\r\nJob Type...
1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ - 04)\r\nStrong Python experience in API development (REST/RPC).\r\nEx...
2,Machine Learning,"Data Scientist (Contractor)\r\n\r\nBangalore, IN\r\n\r\nResponsibilities\r\n\r\nWe are looking for a capable data sc..."
3,iOS Developer,JOB DESCRIPTION:\r\n\r\nStrong framework outside of iOS is always a plus\r\n\r\niOS experience and generalist engine...
4,Full Stack Developer,job responsibility full stack engineer – react role make impact petsmart transforming engineering team meet need rap...


In [7]:
# Quick look at one full posting so we know what the model is being handed
print(df.loc[0, "Job_Title"])
print("-" * 80)
print(df.loc[0, "Job_Description"][:1500], "...")

Flutter Developer
--------------------------------------------------------------------------------
We are looking for hire experts flutter developer. So you are eligible this post then apply your resume.
Job Types: Full-time, Part-time
Salary: ₹20,000.00 - ₹40,000.00 per month
Benefits:
Flexible schedule
Food allowance
Schedule:
Day shift
Supplemental Pay:
Joining bonus
Overtime pay
Experience:
total work: 1 year (Preferred)
Housing rent subsidy:
Yes
Industry:
Software Development
Work Remotely:
Temporarily due to COVID-19 ...


In [8]:
# Descriptions can be very long; truncate so each call fits inside num_ctx (8192 tokens)
# with room for the few-shot prompt and the response. ~4 chars per token, so 4000 chars is
# roughly 1000 tokens of description — comfortably covers the requirements section of a
# typical posting. Raise this only if you also raise num_ctx; on a local model, a larger
# context costs real RAM and slows every call down.
MAX_DESC_CHARS = 4000


def clean_description(text: str, limit: int = MAX_DESC_CHARS) -> str:
    text = re.sub(r"\s+", " ", str(text)).strip()
    return text[:limit]


df["Job_Description_Clean"] = df["Job_Description"].apply(clean_description)
df[["Job_Title", "Job_Description_Clean"]].head(3)

,Job_Title,Job_Description_Clean
0,Flutter Developer,We are looking for hire experts flutter developer. So you are eligible this post then apply your resume. Job Types: ...
1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ - 04) Strong Python experience in API development (REST/RPC). Experien...
2,Machine Learning,"Data Scientist (Contractor) Bangalore, IN Responsibilities We are looking for a capable data scientist to join the A..."


## Step 2 — Job category classification (10 marks)

A fixed taxonomy plus three few-shot examples. Three things make this robust:

1. **Closed label set** in the system message — the model is told to pick from the list only.
2. **Few-shot examples** showing the exact output format (bare label, no explanation).
3. **Post-normalisation** — if the model still returns something off-list ("IT", "Tech"),
   `normalise_category()` fuzzy-matches it back onto the taxonomy and falls back to `"Other"`.

In [9]:
CATEGORIES = [
    "Technology/IT",
    "Finance",
    "Marketing/Sales",
    "Healthcare",
    "Education",
    "Human Resources",
    "Engineering/Manufacturing",
    "Operations/Logistics",
    "Legal",
    "Design/Creative",
    "Other",
]

CATEGORY_LIST_STR = ", ".join(CATEGORIES)

classification_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an expert job-market analyst who classifies job postings into broad domains.\n"
     "Choose EXACTLY ONE category from this list and nothing else:\n"
     f"{CATEGORY_LIST_STR}\n\n"
     "Rules:\n"
     "- Respond with the category label only. No explanation, no punctuation, no quotes.\n"
     "- Judge by the actual work described, not by the employer's industry. A software engineer "
     "at a bank is Technology/IT, not Finance.\n"
     "- Use 'Other' only when the posting genuinely fits none of the categories."),

    # --- few-shot examples ---
    ("human", "Job Title: Senior Data Engineer\nDescription: Build and maintain ETL pipelines in Spark and Airflow, manage the AWS data lake, partner with analytics teams."),
    ("ai", "Technology/IT"),

    ("human", "Job Title: Registered Nurse - ICU\nDescription: Provide direct patient care in the intensive care unit, monitor vitals, administer medication, coordinate with attending physicians."),
    ("ai", "Healthcare"),

    ("human", "Job Title: Financial Analyst\nDescription: Build three-statement models, support quarterly budgeting and variance analysis, prepare board reporting packs."),
    ("ai", "Finance"),

    # --- actual input ---
    ("human", "Job Title: {job_title}\nDescription: {job_description}"),
])

classification_chain = classification_prompt | llm_classify | StrOutputParser()


def normalise_category(raw: str) -> str:
    """Snap whatever the model returned back onto the allowed taxonomy."""
    if not raw:
        return "Other"
    cleaned = raw.strip().strip('".\'').split("\n")[0].strip()

    for cat in CATEGORIES:                                  # exact / case-insensitive match
        if cleaned.lower() == cat.lower():
            return cat
    for cat in CATEGORIES:                                  # substring match ("IT" -> Technology/IT)
        if cleaned.lower() in cat.lower() or cat.lower() in cleaned.lower():
            return cat
    close = difflib.get_close_matches(cleaned, CATEGORIES, n=1, cutoff=0.6)   # fuzzy fallback
    return close[0] if close else "Other"


def classify_job(title: str, description: str) -> str:
    raw = run_with_retry(
        classification_chain.invoke,
        {"job_title": title, "job_description": clean_description(description)},
        label="classify",
    )
    return normalise_category(raw) if raw else "Other"

In [10]:
# --- Expected Output: works on a sample datapoint ---
sample = df.iloc[0]
sample_category = classify_job(sample["Job_Title"], sample["Job_Description"])

print("Job Title :", sample["Job_Title"])
print("Predicted :", sample_category)

Job Title : Flutter Developer
Predicted : Technology/IT


In [11]:
# Sanity check on a few more rows, including the normaliser's behaviour on messy output
for messy in ["IT", "  Finance.  ", "technology / it", "Banking", ""]:
    print(f"{messy!r:25} -> {normalise_category(messy)}")

'IT'                      -> Technology/IT
'  Finance.  '            -> Finance
'technology / it'         -> Technology/IT
'Banking'                 -> Other
''                        -> Other


## Step 3 — Requirements extraction (30 marks)

### 3a. Composite structured-output chain (primary approach)

One call per posting returns all three fields at once. The Pydantic schema is passed to the
model as a tool definition via `with_structured_output`, so the response is validated into a
typed object — no JSON-string parsing, no `json.loads` failures on stray markdown fences.

In [12]:
class JobRequirements(BaseModel):
    """Structured requirements extracted from a single job description."""

    skills: List[str] = Field(
        default_factory=list,
        description=(
            "Specific skills, programming languages, tools, platforms or domain knowledge "
            "explicitly mentioned (e.g. Python, SQL, Tableau, project management, CRM software). "
            "Return concrete named skills only — not soft filler like 'team player'. "
            "Empty list if none are stated."
        ),
    )
    education: str = Field(
        default="Not specified",
        description=(
            "Minimum education level required or preferred, e.g. \"Bachelor's degree in Computer "
            "Science\", \"MBA\", \"High school diploma\". Use exactly 'Not specified' if the "
            "description states no education requirement."
        ),
    )
    experience: str = Field(
        default="Not specified",
        description=(
            "Required experience as stated, e.g. '3+ years', '5-7 years in digital marketing', "
            "'Senior-level'. Use exactly 'Not specified' if no experience requirement is stated."
        ),
    )


extraction_system = (
    "You extract structured hiring requirements from job descriptions.\n"
    "Extract ONLY what is explicitly stated in the description — never infer, never guess, "
    "never fill gaps with what is typical for the role.\n"
    "If a field is not mentioned, return 'Not specified' (or an empty list for skills)."
)

extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", extraction_system),
    ("human",
     "Extract the required skills, education level and experience from the job description below.\n\n"
     "Job Title: {job_title}\n"
     "Job Description:\n{job_description}"),
])

# Two possible paths, and with a local model you can't tell which works by construction alone —
# ChatOllama accepts with_structured_output() for any model, but a model without tool-calling
# support only fails at invoke time. So: build the primary chain, smoke-test it on a tiny fake
# description, and fall back automatically if it errors.

structured_chain = extraction_prompt | llm.with_structured_output(JobRequirements)

# Fallback: a JSON-mode instance (again a constructor field, not a bind) plus schema
# instructions in the prompt, parsed with Pydantic
llm_json = ChatOllama(
    model=OLLAMA_MODEL,
    temperature=0,
    num_ctx=8192,
    format="json",
    keep_alive="30m",
    base_url=OLLAMA_HOST,
)

parser = PydanticOutputParser(pydantic_object=JobRequirements)
fallback_chain = (
    ChatPromptTemplate.from_messages([
        ("system", extraction_system + "\n\nRespond with a single JSON object and nothing else.\n{format_instructions}"),
        ("human", "Job Title: {job_title}\nJob Description:\n{job_description}"),
    ]).partial(format_instructions=parser.get_format_instructions())
    | llm_json
    | parser
)

SMOKE_TEST = {
    "job_title": "Data Analyst",
    "job_description": "We need a Data Analyst with strong SQL and Python skills. "
                       "Bachelor's degree in Statistics required. 3+ years of experience.",
}

try:
    _probe = structured_chain.invoke(SMOKE_TEST)
    extraction_chain, EXTRACTION_MODE = structured_chain, "structured_output (tool calling)"
except Exception as e:
    print(f"Structured output unavailable ({type(e).__name__}: {e})\nFalling back to JSON mode + PydanticOutputParser.")
    _probe = fallback_chain.invoke(SMOKE_TEST)
    extraction_chain, EXTRACTION_MODE = fallback_chain, "json mode + pydantic parser"

print("Extraction mode:", EXTRACTION_MODE)
print("Smoke test result:", _probe)

Extraction mode: structured_output (tool calling)
Smoke test result: skills=['SQL', 'Python'] education="Bachelor's degree in Statistics" experience='3+ years'


In [13]:
NOT_SPECIFIED = "Not specified"


def _blank(value) -> bool:
    return value is None or str(value).strip().lower() in {
        "", "none", "n/a", "na", "null", "not mentioned", "not stated", "not specified"
    }


def extract_requirements(title: str, description: str) -> dict:
    """Return a dict with Required_Skills (str), Skills_List, Education_Required, Experience_Required."""
    result = run_with_retry(
        extraction_chain.invoke,
        {"job_title": title, "job_description": clean_description(description)},
        label="extract",
    )

    if result is None:                     # all retries failed — degrade gracefully
        return {
            "Required_Skills": NOT_SPECIFIED,
            "Skills_List": [],
            "Education_Required": NOT_SPECIFIED,
            "Experience_Required": NOT_SPECIFIED,
        }

    # De-duplicate skills case-insensitively while preserving order
    seen, skills = set(), []
    for s in (result.skills or []):
        s = str(s).strip()
        if s and s.lower() not in seen:
            seen.add(s.lower())
            skills.append(s)

    return {
        "Required_Skills": ", ".join(skills) if skills else NOT_SPECIFIED,
        "Skills_List": skills,
        "Education_Required": NOT_SPECIFIED if _blank(result.education) else str(result.education).strip(),
        "Experience_Required": NOT_SPECIFIED if _blank(result.experience) else str(result.experience).strip(),
    }

In [14]:
# --- Expected Output: works on a sample datapoint ---
sample_reqs = extract_requirements(sample["Job_Title"], sample["Job_Description"])
print(json.dumps({k: v for k, v in sample_reqs.items() if k != "Skills_List"}, indent=2))

{
  "Required_Skills": "Not specified",
  "Education_Required": "Not specified",
  "Experience_Required": "Not specified"
}


### 3b. Sub-task prompts (the one-by-one alternative)

The assignment allows either a composite prompt or three separate ones. The composite chain
above is what the pipeline actually uses (1 call instead of 3 — 3× cheaper and faster), but
here are the individual chains to show the decomposed version working.

In [15]:
skills_chain = (
    ChatPromptTemplate.from_template(
        "List the key skills and technologies explicitly mentioned in the job description below.\n"
        "Return a comma-separated list only. If none are mentioned, return 'Not specified'.\n\n"
        "{job_description}"
    ) | llm | StrOutputParser()
)

education_chain = (
    ChatPromptTemplate.from_template(
        "What is the minimum education level required or preferred for this job?\n"
        "Answer in a short phrase only. If the description does not state one, return 'Not specified'.\n\n"
        "{job_description}"
    ) | llm | StrOutputParser()
)

experience_chain = (
    ChatPromptTemplate.from_template(
        "What experience does this job require (years and/or seniority level)?\n"
        "Answer in a short phrase only. If the description does not state any, return 'Not specified'.\n\n"
        "{job_description}"
    ) | llm | StrOutputParser()
)

_desc = clean_description(sample["Job_Description"])
print("Skills     :", skills_chain.invoke({"job_description": _desc}).strip())
print("Education  :", education_chain.invoke({"job_description": _desc}).strip())
print("Experience :", experience_chain.invoke({"job_description": _desc}).strip())

Skills     : Flutter, software development
Education  : 1 year (Preferred)
Experience : 1 year (Preferred)


## Step 4 — Apply the chains to every posting (10 marks)

Two LLM calls per posting (classification + extraction) = 50 calls for 25 postings.

**Expect this to take a while.** On a 7–8B model: roughly 3–8s per call on a decent GPU,
15–40s per call on CPU only. So anywhere from ~4 minutes to ~30 minutes for the full run.
The per-row print statements let you watch progress rather than stare at a blank cell.

No sleep between calls — there are no rate limits on a local server, and the model is already
the bottleneck. If the machine gets uncomfortably hot on a long run, raise it.

In [17]:
SLEEP_BETWEEN_CALLS = 0

records = []
start = time.time()

for idx, row in df.iterrows():
    title, description = row["Job_Title"], row["Job_Description"]
    print(f"[{idx + 1:>2}/{len(df)}] {str(title)[:60]}")

    category = classify_job(title, description)          # Step 2 chain
    reqs = extract_requirements(title, description)      # Step 3 chain

    records.append({
        "Predicted_Category": category,
        "Required_Skills": reqs["Required_Skills"],
        "Skills_List": reqs["Skills_List"],
        "Education_Required": reqs["Education_Required"],
        "Experience_Required": reqs["Experience_Required"],
    })

    print(f"       -> {category} | skills: {len(reqs['Skills_List'])} | "
          f"edu: {reqs['Education_Required'][:35]} | exp: {reqs['Experience_Required'][:25]}")
    time.sleep(SLEEP_BETWEEN_CALLS)

elapsed = time.time() - start
print(f"\nProcessed {len(records)} postings in {elapsed:.1f}s ({elapsed / max(len(records), 1):.1f}s per posting)")

[ 1/25] Flutter Developer
       -> Technology/IT | skills: 0 | edu: Not specified | exp: Not specified
[ 2/25] Django Developer
       -> Technology/IT | skills: 10 | edu: Not specified | exp: Not specified
[ 3/25] Machine Learning
       -> Technology/IT | skills: 6 | edu: Graduate or M.Sc. in Computer Scien | exp: At least 3 years
[ 4/25] iOS Developer
       -> Technology/IT | skills: 7 | edu: Not specified | exp: 1+ years
[ 5/25] Full Stack Developer
       -> Technology/IT | skills: 0 | edu: Not specified | exp: 5+ years, 2+ years (recen
[ 6/25] Java Developer
       -> Technology/IT | skills: 0 | edu: Bachelor's Degree in Computer Scien | exp: 2 years (Required) Softwa
[ 7/25] Full Stack Developer
       -> Technology/IT | skills: 0 | edu: B.Sc degree in Computer Science or  | exp: Minimum 2 years of experi
[ 8/25] JavaScript Developer
       -> Technology/IT | skills: 7 | edu: Any graduation, Any PG, Any Doctora | exp: 3 - 8 years
[ 9/25] DevOps Engineer
       -> Technology/IT

KeyboardInterrupt: 

In [18]:
results_df = pd.DataFrame(records)
print("Results shape:", results_df.shape)
results_df.head()

Results shape: (22, 5)


,Predicted_Category,Required_Skills,Skills_List,Education_Required,Experience_Required
0,Technology/IT,Not specified,[],Not specified,Not specified
1,Technology/IT,"Python, Django, API development, REST, RPC, Linux, SQL, JSON, PyUnit, Automated unit testing","[Python, Django, API development, REST, RPC, Linux, SQL, JSON, PyUnit, Automated unit testing]",Not specified,Not specified
2,Technology/IT,"Python, Java, Machine Learning, Deep Learning, Statistics, Applied Mathematics","[Python, Java, Machine Learning, Deep Learning, Statistics, Applied Mathematics]","Graduate or M.Sc. in Computer Science, Mathematics or equivalent",At least 3 years
3,Technology/IT,"Objective-C, Cocoa Touch, Core Data, Core Animation, Core Graphics, Core Text, third-party libraries and APIs","[Objective-C, Cocoa Touch, Core Data, Core Animation, Core Graphics, Core Text, third-party libraries and APIs]",Not specified,1+ years
4,Technology/IT,Not specified,[],Not specified,"5+ years, 2+ years (recent experience working React)"


**If it's too slow.** Options, roughly in order of payoff:

1. Use a smaller model (`qwen2.5:3b`, `llama3.2:3b`). Expect noticeably weaker extraction —
   check whether the JSON still validates before committing to it.
2. Run a quantised variant (`qwen2.5:7b-instruct-q4_K_M`) — much lighter with modest quality loss.
3. Drop `MAX_DESC_CHARS` to ~2500. Fewer input tokens is the single biggest lever on local latency.
4. Concurrency via `.batch(..., config={"max_concurrency": 2})`. Helps only if you have GPU
   headroom — on one CPU-bound model, parallel calls just contend for the same cores and can
   end up slower than sequential.

Concurrency is left off by default for that reason.

## Step 5 — Update the DataFrame with the new columns (5 marks)

In [19]:
df_final = pd.concat([df.drop(columns=["Job_Description_Clean"]), results_df], axis=1)

# Column order: originals first, then everything the LLM produced
df_final = df_final[[
    "Job_Title",
    "Job_Description",
    "Predicted_Category",
    "Required_Skills",
    "Education_Required",
    "Experience_Required",
    "Skills_List",
]]

print("Final shape:", df_final.shape)
df_final[["Job_Title", "Predicted_Category", "Required_Skills",
          "Education_Required", "Experience_Required"]]

Final shape: (25, 7)


,Job_Title,Predicted_Category,Required_Skills,Education_Required,Experience_Required
0,Flutter Developer,Technology/IT,Not specified,Not specified,Not specified
1,Django Developer,Technology/IT,"Python, Django, API development, REST, RPC, Linux, SQL, JSON, PyUnit, Automated unit testing",Not specified,Not specified
2,Machine Learning,Technology/IT,"Python, Java, Machine Learning, Deep Learning, Statistics, Applied Mathematics","Graduate or M.Sc. in Computer Science, Mathematics or equivalent",At least 3 years
3,iOS Developer,Technology/IT,"Objective-C, Cocoa Touch, Core Data, Core Animation, Core Graphics, Core Text, third-party libraries and APIs",Not specified,1+ years
4,Full Stack Developer,Technology/IT,Not specified,Not specified,"5+ years, 2+ years (recent experience working React)"
5,Java Developer,Technology/IT,Not specified,"Bachelor's Degree in Computer Science, Information Systems, or related field, or combination of education and equiva...",2 years (Required) Software development: 2 years
6,Full Stack Developer,Technology/IT,Not specified,B.Sc degree in Computer Science or Engineering,"Minimum 2 years of experience developing NodeJS, Java, and NoSQL solutions (MongoDB, Elasticsearch, Redis) with expe..."
7,JavaScript Developer,Technology/IT,"ReactJS, NodeJS, Azure Functions, GraphQL, HTML5, CSS3, JavaScript","Any graduation, Any PG, Any Doctorate",3 - 8 years
8,DevOps Engineer,Technology/IT,Not specified,Not specified,"Senior level experience with automation tools, scripting experience with Bash, Ruby, Python, Java, and experience wi..."
9,Software Engineer,Technology/IT,Not specified,"BS or MS; computer engineering, computer science or related technical field",Minimum 7 years of software development experience


### Verification / spot-checks

Read the description alongside the extraction for two or three rows and confirm the fields
actually appear in the text. Note in the write-up any row where the model over-reached.

In [20]:
for i in [0, 7, 15]:
    if i >= len(df_final):
        continue
    r = df_final.iloc[i]
    print("=" * 90)
    print("TITLE      :", r["Job_Title"])
    print("CATEGORY   :", r["Predicted_Category"])
    print("SKILLS     :", r["Required_Skills"][:200])
    print("EDUCATION  :", r["Education_Required"])
    print("EXPERIENCE :", r["Experience_Required"])
    print("-" * 90)
    print("DESCRIPTION:", clean_description(r["Job_Description"], 700), "...")
    print()

TITLE      : Flutter Developer
CATEGORY   : Technology/IT
SKILLS     : Not specified
EDUCATION  : Not specified
EXPERIENCE : Not specified
------------------------------------------------------------------------------------------
DESCRIPTION: We are looking for hire experts flutter developer. So you are eligible this post then apply your resume. Job Types: Full-time, Part-time Salary: ₹20,000.00 - ₹40,000.00 per month Benefits: Flexible schedule Food allowance Schedule: Day shift Supplemental Pay: Joining bonus Overtime pay Experience: total work: 1 year (Preferred) Housing rent subsidy: Yes Industry: Software Development Work Remotely: Temporarily due to COVID-19 ...

TITLE      : JavaScript Developer
CATEGORY   : Technology/IT
SKILLS     : ReactJS, NodeJS, Azure Functions, GraphQL, HTML5, CSS3, JavaScript
EDUCATION  : Any graduation, Any PG, Any Doctorate
EXPERIENCE : 3 - 8 years
------------------------------------------------------------------------------------------
DESCRIPTION: J

In [21]:
# Coverage summary — useful evidence of quality for the report
print("Category distribution:")
print(df_final["Predicted_Category"].value_counts().to_string())

total = len(df_final)
print("\nField coverage (share where the model found something):")
for col in ["Required_Skills", "Education_Required", "Experience_Required"]:
    found = (df_final[col] != NOT_SPECIFIED).sum()
    print(f"  {col:22} {found}/{total}  ({found / total:.0%})")

# Distinguish "the model read the posting and found nothing" from "the call failed".
# Both end up as 'Not specified' / 'Other' in the table, so check this before trusting coverage.
if CALL_FAILURES:
    print(f"\n!! {len(CALL_FAILURES)} call(s) failed outright — those rows are placeholders, not findings:")
    for f in CALL_FAILURES[:5]:
        print("   ", f["label"], "-", f["error"])
else:
    print("\nNo failed calls — every 'Not specified' is a genuine extraction result.")

print("\nMost common skills across all 25 postings:")
all_skills = pd.Series([s.lower() for lst in df_final["Skills_List"] for s in lst])
print(all_skills.value_counts().head(15).to_string())

Category distribution:
Predicted_Category
Technology/IT      21
Design/Creative     1

Field coverage (share where the model found something):
  Required_Skills        11/25  (44%)
  Education_Required     17/25  (68%)
  Experience_Required    21/25  (84%)

No failed calls — every 'Not specified' is a genuine extraction result.

Most common skills across all 25 postings:


TypeError: 'float' object is not iterable

### Export

In [22]:
export_df = df_final.drop(columns=["Skills_List"])
export_df.to_csv("job_postings_analyzed.csv", index=False)
export_df.to_json("job_postings_analyzed.json", orient="records", indent=2)

print("Saved job_postings_analyzed.csv and job_postings_analyzed.json\n")
print("Sample of the JSON output:\n")
print(export_df.head(2).to_json(orient="records", indent=2))

Saved job_postings_analyzed.csv and job_postings_analyzed.json

Sample of the JSON output:

[
  {
    "Job_Title":"Flutter Developer",
    "Job_Description":"We are looking for hire experts flutter developer. So you are eligible this post then apply your resume.\r\nJob Types: Full-time, Part-time\r\nSalary: \u20b920,000.00 - \u20b940,000.00 per month\r\nBenefits:\r\nFlexible schedule\r\nFood allowance\r\nSchedule:\r\nDay shift\r\nSupplemental Pay:\r\nJoining bonus\r\nOvertime pay\r\nExperience:\r\ntotal work: 1 year (Preferred)\r\nHousing rent subsidy:\r\nYes\r\nIndustry:\r\nSoftware Development\r\nWork Remotely:\r\nTemporarily due to COVID-19",
    "Predicted_Category":"Technology\/IT",
    "Required_Skills":"Not specified",
    "Education_Required":"Not specified",
    "Experience_Required":"Not specified"
  },
  {
    "Job_Title":"Django Developer",
    "Job_Description":"PYTHON\/DJANGO (Developer\/Lead) - Job Code(PDJ - 04)\r\nStrong Python experience in API development (REST\/RPC)

## Notes for the write-up

- **Why a local model** — no API key, no per-call cost, and the job descriptions never leave the
  machine, which matters for scraped third-party data. The trade-off is instruction-following:
  a 7B local model needs tighter prompts and more defensive post-processing than a hosted
  frontier model, which is exactly what the normaliser and the JSON fallback are there for.
- **Why `num_ctx=8192`** — Ollama defaults to a 2048-token context and silently discards
  overflow rather than erroring. With a full job description in the prompt, the dropped portion
  is the system instructions, and the model returns confident nonsense. Setting it explicitly is
  the single most important configuration line in the notebook.
- **Why temperature 0** — extraction and classification are deterministic tasks; sampling would
  make the same posting yield different labels across runs and make results unreproducible.
- **Why a closed label set + normaliser** — free-text category output drifts ("Tech", "IT",
  "Information Technology"), which fragments any downstream aggregation. Constraining the
  vocabulary at both the prompt and the post-processing layer guarantees clean joins.
- **Why structured output over regex** — the schema is enforced by the model's tool-calling /
  JSON-schema layer, so malformed JSON and markdown code fences never reach the parser. The
  smoke test in Step 3 picks the working path automatically, since tool-calling support varies
  from one local model to the next.
- **Handling missing information** — the prompt instructs extract-only-what-is-stated and the
  post-processor maps every empty/`N/A`/`null` variant onto a single `"Not specified"` sentinel,
  so the column has one canonical missing value rather than five.
- **Known limitations** — descriptions are truncated to 4000 characters, so a requirement buried
  at the very end of a long posting can be missed; the category taxonomy is fixed and dataset-
  dependent; there is no gold-labelled ground truth, so accuracy is assessed by spot-check rather
  than measured; and a 7B local model is weaker at boundary cases than a hosted large model —
  expect it to occasionally list a soft skill among the technical ones, or read "preferred" as
  "required". Worth flagging one or two such cases from your spot-checks in the report.